[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/thin_lens/thin_lens.ipynb)

# Thin Lenses as Maps on Rays

This example traces rays through thin lenses, a prism and a mirror. A thin lens bends every ray that crosses it by an amount set by how far from the lens's centre it crosses: rays from one point leave it aimed at one image point, and parallel rays leave it aimed at its focus. A train of several elements still sends the rays from one point through one image, wherever the elements stand and however they are tilted.

A ray is a line, the join of two points, and a thin lens at the origin is a map on lines: `Line - (Line & origin) * (mv.x / focal)`, each line sheared by its incidence with the lens's centre. A lens elsewhere is that map carried by a motor, `motor >> lens(motor << Line)`, and a train of elements is the composition of their maps, one map acting on the whole bundle. Lines are global, so the free space between elements needs no map of its own, and an image is the meet of two rays that leave the train.

In ray transfer matrix notation a ray reads as its height and slope where it crosses one reference plane, each element as an ABCD matrix on that pair, and the free space between elements as a propagation matrix of its own.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from functools import reduce
from itertools import accumulate

import numpy as np

from numga import stack
from numga import NumpyContext
from numga.algebras import PGA2D as ga
from examples.util.animation import save_figure, show_animation
from examples.optics.thin_lens import render

mv = NumpyContext(ga).multivector
# A ray is the join of two points; the plane of an element is a vector, as a mirror is.
Line = ga.gatype.antibivector()                                                 # Antibivector-
Plane = ga.gatype.vector()                                                      # Vector+
# Where two lines cross: a meet, oriented by what crosses it.
Crossing = ga.gatype.bivector()                                                 # Bivector+
LineMap = ga.gatype((Line, Line))
Motor = ga.gatype.rotor()                                                       # Rotor+
origin = mv.point([0, 0])                                                       # [] Point-

## 1. One lens, and two

A lens anywhere is the lens at the origin, carried onto its plane by the translator that moves the origin there. Rays from an object point through the lens all pass through one point after it, the image, found as the meet of any two of them; its distance from the lens obeys the thin lens equation. Two lenses compose into one map, `second_lens(first_lens)`, and parallel rays through both meet at the system's back focal point, where Gullstrand's formula puts it.

In [ ]:
# Where each lens stands along x and its focal length; the subject; the pupil's half-height, in rays.
first_station, first_focal = 1.5, 1.0
second_station, second_focal = 2.1, 0.5
subject = mv.point([-2.0, 0.5])                                                 # [] Point-
pupil_height, n_rays = 0.8, 9

In [ ]:
def thin_lens(plane: Plane, focal: float) -> LineMap:
    """A thin lens in a vertical plane, as a map on lines.

    At the origin it shears a line's x coefficient by the line's incidence with the origin over
    the focal length. Anywhere else it is that map conjugated by the translator onto its plane,
    generated by the ideal point along y scaled by the plane's offset from the origin.
    """
    shear = Line - (Line & origin) * (mv.x / focal)                             # [] Line- <- Line-
    shift = (mv.wx * (plane & origin) * 0.5).exp()                              # [] Motor+
    return shift >> shear(shift << Line)


first_plane, second_plane = mv.x - mv.w * first_station, mv.x - mv.w * second_station   # [] Plane+, Plane+
first_lens = thin_lens(first_plane, first_focal)                               # [] Line- <- Line-
second_lens = thin_lens(second_plane, second_focal)                            # [] Line- <- Line-
# Rays from the subject through a pupil of points on the first lens's plane, about the optical axis.
axis = origin & mv.direction([1, 0])                                           # [] Line-
pupil = (first_plane ^ axis) + mv.direction([0, 1]) * np.linspace(-pupil_height, pupil_height, n_rays)   # [rays] Point-
rays = subject & pupil                                                         # [rays] Line-
refracted = first_lens(rays)                                                   # [rays] Line-
image = refracted[0] ^ refracted[1]                                            # [] Crossing+
# Two lenses: one map. Parallel rays, joins with the point at infinity along x, meet after it at the focus.
system = second_lens(first_lens)                                               # [] Line- <- Line-
parallel = mv.direction([1, 0]) & pupil                                        # [rays] Line-
between_lenses = first_lens(parallel)                                          # [rays] Line-
focused = system(parallel)                                                     # [rays] Line-
focus = focused[0] ^ focused[1]                                                # [] Crossing+
imaging = stack([rays, refracted])                                             # [legs, rays] Line-
focusing = stack([parallel, between_lenses, focused])                          # [legs, rays] Line-
save_figure(render.draw_lenses(subject, imaging, image, focusing, focus, stack([first_plane, second_plane])), "thin_lens");

## 2. A train of elements, moving

Every element is a map on lines defined in the plane `mv.x`: a thin lens shears a line by its incidence with the centre, a thin prism by its incidence with a point at infinity, the same turn at every height, and a flat mirror is the sandwich by its plane. A motor places each, and the train is their composition, built element by element as the bundle passes through. The first lens slides and tilts and the mirror rocks; a tilted ideal lens still images, since concurrent rays stay concurrent, and the image, the meet of two outgoing rays, moves with them.

In [ ]:
# The train's subject and the point its fan of rays aims at, the fan's half-height in rays; where each
# element stands along x; how far the first lens slides and tilts and the mirror rocks, about its tilt at
# rest; the frames, and their duration in milliseconds.
train_subject = mv.point([-1.0, 0.5])                                           # [] Point-
aim = mv.point([1.0, 0.0])                                                      # [] Point-
fan_height, n_fan_rays = 0.6, 7
stations = np.array([1.0, 1.9, 2.2, 3.2])
slide, lens_tilt, rock, mirror_tilt = 0.3, 0.3, 0.1, np.pi / 4
n_frames, frame_duration = 108, 40

In [ ]:
# Every train element is defined in the plane x == 0.
home = mv.x                                                                    # [] Plane+
# An optical train in the abstract. Every element is a map on lines defined in the home plane:
# a thin lens shears a line by its incidence with the centre, a thin prism by its incidence
# with an ideal point (the same slope change for every height), and a flat mirror is the
# sandwich by the home plane.
elements = (                                                                   # [elements] Line- <- Line-
    # A thin lens of focal length 1.
    Line - (Line & origin) * (home / 1.0),
    # A thin prism, slope change 0.15.
    Line - (Line & mv.direction([0, 1])) * home * 0.15,
    # A thin lens of focal length -2.
    Line - (Line & origin) * (home / -2.0),
    # A flat mirror in the home plane.
    home.normalized() >> Line,
)


def trace(rays: Line, motors: tuple[Motor, ...], elements: tuple[LineMap, ...]) -> tuple[Plane, Line, LineMap]:
    """An optical train with each element placed by its motor, and a bundle passing through it.

    The result is the element planes [elements], the bundle before and after each element
    [elements + 1, rays], and the train: the composition of the placed elements, one map on
    lines that acts on the whole bundle.
    """
    # Each element conjugated into place.
    placed = [motor >> element(motor << Line) for motor, element in zip(motors, elements)]   # [elements] Line- <- Line-
    # The bundle after each element, and the train as one map, each placed element applied to the last.
    legs = stack(list(accumulate(placed, lambda bundle, element: element(bundle), initial=rays)))   # [elements + 1, rays] Line-
    train = reduce(lambda composed, element: element(composed), placed, Line)  # [] Line- <- Line-
    return stack([motor >> home for motor in motors]), legs, train


fan = train_subject & (aim + mv.direction([0, 1]) * np.linspace(-fan_height, fan_height, n_fan_rays))   # [rays] Line-
# Each element's motor onto its station.
stops = (mv.wx * (-stations / 2)).exp()                                        # [elements] Motor+


def frames():
    """Per frame the subject, the element planes, the bundle before and after each element, the train as
    one map, and the image, the meet of two outgoing rays."""
    for phase in np.linspace(0.0, 2 * np.pi, n_frames, endpoint=False):
        # The first lens slides and tilts, and the mirror rocks about its pivot; the prism and the second lens are fixed.
        lens_slide = (mv.wx * (-slide * np.sin(phase) / 2)).exp()              # [] Motor+
        lens_turn = (mv.xy * (lens_tilt * np.sin(2 * phase) / 2)).exp()        # [] Motor+
        mirror_turn = (mv.xy * ((mirror_tilt + rock * np.cos(phase)) / 2)).exp()   # [] Motor+
        motors = (lens_slide * stops[0] * lens_turn, stops[1], stops[2], stops[3] * mirror_turn)
        planes, legs, composed = trace(fan, motors, elements)
        outgoing = composed(fan)                                               # [rays] Line-
        yield train_subject, planes, legs, composed, outgoing[0] ^ outgoing[-1]


show_animation(render.animate_train(frames()), "thin_lens", frame_duration)

In [ ]:
# checks
# Every ray through one lens passes through the image, at the distance the thin lens equation gives: a point's
# signed distance from the lens is its incidence with the lens plane over its weight. Two lenses focus parallel
# rays on the axis where Gullstrand's formula puts it; the train images in every frame.
object_distance = -(first_plane & subject) / (mv.w & subject)                  # [] Scalar+
image_distance = (first_plane & image) / (mv.w & image)                        # [] Scalar+
gap = (first_plane - second_plane) & origin                                    # [] Scalar+
effective = 1 / (1 / first_focal + 1 / second_focal - gap / (first_focal * second_focal))
np.testing.assert_allclose((refracted ^ image).kernel, 0.0, atol=1e-11)
np.testing.assert_allclose((1 / object_distance + 1 / image_distance).to_array(), 1 / first_focal, atol=1e-12)
np.testing.assert_allclose(((second_plane & focus) / (mv.w & focus)).to_array(), (effective * (first_focal - gap) / first_focal).to_array(), atol=1e-12)
np.testing.assert_allclose((mv.y ^ focus).kernel, 0.0, atol=1e-12)
for _, _, legs, composed, picture in frames():
    np.testing.assert_allclose((composed(legs[0]) ^ picture).kernel, 0.0, atol=1e-10)